# 02. 국제 커피 가격 추세 & 수입단가 교차검증 (Section 3)

**목적**: FRED(IMF Primary Commodity Prices)의 국제 커피 가격(Arabica/Robusta) 추세를 보고,
앞서 Section 2에서 발견한 **한국 수입 단가 급등이 국제 시세로 설명되는지** 교차검증한다.

- 입력: `data/raw/coffee_prices_fred_2024_2026.csv`, `data/processed/monthly_import_summary.csv`
- 출력: `data/processed/price_vs_import_unitprice.csv`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pr = pd.read_csv("../data/raw/coffee_prices_fred_2024_2026.csv")
pr["date"] = pd.to_datetime(pr["year_month"], format="%Y-%m")
pr.head()

## 1. 국제 커피 가격 추세 (USD/kg)

FRED 원자료는 US 센트/lb. `fetch_prices.py`에서 USD/kg로 환산해 둠
(USD/kg = 센트/lb × 0.0220462). 우리 수입 단가(USD/kg)와 같은 단위로 비교하기 위함.

In [ ]:
x = pr["date"]
plt.figure(figsize=(11, 5))
plt.plot(x, pr["arabica_otm_usd_per_kg"], marker="o", color="#b5651d", label="Arabica (Other Milds)")
plt.plot(x, pr["robusta_usd_per_kg"], marker="s", color="#6b8e23", label="Robusta")
plt.ylabel("USD / kg")
plt.xlabel("Month")
plt.title("International Coffee Prices (FRED / IMF), USD per kg")
plt.legend()
plt.tight_layout()
plt.show()

**관찰**: Arabica는 2024-01 ~\$4.5/kg → 2025년 \$9/kg 부근까지 급등 후 \$7대로 일부 조정.
Robusta는 \$3~5대에서 등락. 두 시세 모두 2024~25년 구조적 상승(브라질 가뭄·베트남 공급난).

## 2. 한국 수입 단가 vs 국제 시세 교차검증

Section 2의 월별 수입 단가(전 원산지 평균)를 국제 시세 위에 겹쳐 본다.

In [ ]:
imp = pd.read_csv("../data/processed/monthly_import_summary.csv")[
    ["year_month", "avg_unit_price_usd_per_kg"]
].rename(columns={"avg_unit_price_usd_per_kg": "korea_import_usd_per_kg"})

m = imp.merge(
    pr[["year_month", "arabica_otm_usd_per_kg", "robusta_usd_per_kg"]],
    on="year_month", how="inner",
)
m.to_csv("../data/processed/price_vs_import_unitprice.csv", index=False, encoding="utf-8-sig")

corr_arabica = m["korea_import_usd_per_kg"].corr(m["arabica_otm_usd_per_kg"])
corr_robusta = m["korea_import_usd_per_kg"].corr(m["robusta_usd_per_kg"])
print(f"상관계수  한국수입단가 vs Arabica: {corr_arabica:.3f}")
print(f"상관계수  한국수입단가 vs Robusta: {corr_robusta:.3f}")
print(f"평균 격차 (수입단가 - Arabica): {(m.korea_import_usd_per_kg - m.arabica_otm_usd_per_kg).mean():.2f} USD/kg")
m.tail()

In [ ]:
xx = pd.to_datetime(m["year_month"], format="%Y-%m")
plt.figure(figsize=(11, 5))
plt.plot(xx, m["arabica_otm_usd_per_kg"], color="#b5651d", label="Arabica (world)")
plt.plot(xx, m["robusta_usd_per_kg"], color="#6b8e23", label="Robusta (world)")
plt.plot(xx, m["korea_import_usd_per_kg"], color="#1f3a5f", marker="o",
         linewidth=2, label="Korea import unit price (all origins)")
plt.ylabel("USD / kg")
plt.xlabel("Month")
plt.title("Korea Green Bean Import Unit Price vs World Coffee Prices")
plt.legend()
plt.tight_layout()
plt.show()

**결론 (Section 3 핵심)**:
- 한국 수입 단가는 **Arabica와 상관 0.87**, Robusta와는 0.26 → 수입 비용은 **아라비카 시세가 견인**.
- 수입 단가는 아라비카보다 평균 ~\$0.93/kg 낮음 (베트남 로부스타·브라질 내추럴 등 상대적 저가 물량 혼합 반영).
- 2026-03 기준 수입단가 \$7.40/kg ≈ 아라비카 \$7.37/kg 로 사실상 수렴.
- → **수입 단가 2배 상승은 국내 요인이 아니라 국제 아라비카 시세 급등이라는 외생적 비용 충격으로 설명됨.**
  이는 국내 F&B 원가 압박(Section 6)으로 이어진다.
